# 07 — 拷贝数变异推断 (CNV Inference)

用 `infercnvpy`（纯 Python，**不用 R**）从 scRNA-seq 表达数据推断
大规模染色体拷贝数变异（copy number variation, CNV）。

## 为什么用 infercnvpy？

`infercnvpy` 是 R/InferCNV 的 Python 实现，核心算法相同：
以参考正常细胞的平均表达为基线，用滑动窗口平滑基因表达信号，
检测染色体臂级别的大规模偏离——这是肿瘤细胞区别于正常细胞的关键特征之一。
纯 Python 避免了 R 桥接的 IO 开销与依赖脆弱性。

## 为什么做 CNV 推断？

肿瘤细胞区别于正常细胞的核心特征之一是基因组不稳定性——
大片段染色体拷贝数变异（amplifications / deletions）是恶性肿瘤的标志性事件。
scRNA-seq 虽然测的是转录组而非基因组，但染色体级别的大规模 CNV
会系统性地改变整段染色体上数百个基因的表达水平——
这个信号足够强，可以被滑动窗口平均算法从表达数据中检测出来。

**临床意义**：CNV 信号强度是区分恶性上皮细胞与正常上皮细胞的
关键技术指标之一。在胃癌前病变场景中，CNV 谱可以辅助判断：
哪些细胞群体已出现基因组不稳定性（癌前病变进展信号），
哪些细胞群体仍维持二倍体状态（正常或低风险）。

## 工作流


## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（注释），读 `06_annotated_v*.h5ad`
- **下游**：下游分析（恶性细胞鉴定 / 跨组对比），产出 `09_cnv_v*.h5ad` + CNV 热图

### 为什么要迭代回跑？
拷贝数变异推断 (CNV) 的结果是下游分析和 PI 生物学判断的基础。如果在后续分析中发现：
- DEG 阈值过高导致遗漏关键基因、过低导致假阳性
- 通路富集缺少预期应出现的生物学通路
- 调控网络缺少已知的主控转录因子
- CNV 信号不符合病理学预期
可能需要调整本 stage 的参数重新计算。

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`（旧版不覆盖）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改以下参数后重跑本 notebook）：
- `CNV_WINDOW_SIZE` / `CNV_STEP` — 滑动窗口大小与步长
- `CNV_LFC_CLIP` — logFC 裁剪上限
- `REFERENCE_CELL_TYPES` — 参考正常细胞类型列表
- `REFERENCE_KEY` — 细胞类型 obs 列

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为结果可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：下游 notebook 的 `UPSTREAM_PATH` 指向你决定采用的版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"09_cnv"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询「拷贝数变异推断 (CNV) 有哪些版本？哪些依赖 06_annotated_v1？」，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH      — 06 注释结果 h5ad
# OUTPUT_PATH         — 本 stage 产出 checkpoint 路径。
#                        版本号 _v1 与 adata.uns['version'] 保持一致。
#                        如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# REFERENCE_CELL_TYPES — 作为"正常"参考的细胞类型（用于 CNV 基线）
#                       列表中的细胞类型被假定为无大片段 CNV
# REFERENCE_KEY      — obs 列用作参考单元格标记（默认用 cell_type_final_v1）
# CNV_WINDOW_SIZE    — 滑动窗口大小（基因数）
# CNV_STEP           — 滑动步长（基因数）
# CNV_LFC_CLIP       — logFC 上限裁剪
# CNV_MIN_CHR_GENES  — 每条染色体最少基因数（低于此值丢弃该染色体）

UPSTREAM_PATH = "results/06_annotated_v1.h5ad"
OUTPUT_PATH   = "results/09_cnv_v1.h5ad"

REFERENCE_CELL_TYPES = [
    "B cell", "plasma cell", "T cell", "myeloid cell",
    "mast cell", "endothelial cell", "fibroblast", "pericyte",
]
REFERENCE_KEY = "cell_type_final_v1"

CNV_WINDOW_SIZE   = 100
CNV_STEP          = 10
CNV_LFC_CLIP      = 3.0
CNV_MIN_CHR_GENES = 20

In [ ]:
# === setup：sys.path + 导入 + 加载上游 ===
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/07_downstream/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc
_root = os.getcwd()
_root_candidates = [
    _root,
    os.path.abspath(os.path.join(_root, "..")),
    os.path.abspath(os.path.join(_root, "..", "..")),
]

for _cand in _root_candidates:
    if os.path.isdir(os.path.join(_cand, "src", "scrna_integration")):
        _root = _cand
        break
else:
    _root = os.environ.get("PROJECT_ROOT", _root)

if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")
# 导入依赖。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import infercnvpy as cnv
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  infercnvpy {cnv.__version__}")
# 加载上游 06 输出。
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 检查细胞类型列
if REFERENCE_KEY in adata.obs.columns:
    _ct = adata.obs[REFERENCE_KEY].dropna()
    print(f"细胞类型 ({REFERENCE_KEY}): {sorted(_ct.unique())}")
else:
    print(f"WARNING: REFERENCE_KEY '{REFERENCE_KEY}' 不在 obs 列中")
    if "cell_type_final_v1" in adata.obs.columns:
        print("  cell_type_final_v1 可用——请将 REFERENCE_KEY 设为它")
    print(f"  可用 obs 列: {list(adata.obs.columns)}")

# 检查 var 列（基因组位置是否已存在）
_var_cols = list(adata.var.columns)
_pos_cols = ["chromosome", "start", "end"]
_pos_missing = [c for c in _pos_cols if c not in _var_cols]
if _pos_missing:
    print(f"var 缺少基因组位置列: {_pos_missing}——将在后续 cell 注入")
else:
    print(f"var 已包含基因组位置列: {_pos_cols}")
    _valid_chr = adata.var["chromosome"].dropna()
    print(f"  有位置信息的基因: {len(_valid_chr)}/{adata.n_vars}")

## 基因位置注释

用 mygene 为每个基因查询基因组位置（chromosome, start, end）。
**为什么用 mygene？** 它提供了 Ensembl gene ID → 基因组坐标的批量查询 API，
本项目 01 已在用 mygene 做基因 ID 同步，可以复用。

如果网络查询失败（mygene.info 不稳定或无网络），PI 也可以手动从
[Ensembl BioMart](https://www.ensembl.org/biomart/) 下载人类基因位置 CSV
（选 Gene stable ID / Chromosome/scaffold name / Gene start (bp) / Gene end (bp) 四列），
填入 PARAMS 的 `GENE_POS_CSV` 路径。

In [ ]:
# 基因位置注入——优先检测 mygene 在线查询，兜底本地 CSV。
# 逻辑复用自 student-code workflow_for_pseudotime/4.2（重写，去硬编码路径）。

_pos_cols = ["chromosome", "start", "end"]
_pos_need_inject = [c for c in _pos_cols if c not in adata.var.columns]

if _pos_need_inject:
    print(f"需要注入位置列: {_pos_need_inject}")

    # 确定基因 ID 列：优先用 ensembl_id
    _id_col = None
    for _cand in ["ensembl_id", "gene_ids", "gene_id"]:
        if _cand in adata.var.columns:
            _id_col = _cand
            break

    if _id_col is None:
        _sample_ids = adata.var_names[:5].tolist()
        if any(str(g).startswith("ENSG") for g in _sample_ids):
            _id_col = "_index"
            print("检测到 var.index 为 Ensembl gene ID，用作查询键")
        else:
            print("WARNING: 无法确定基因 ID 列，基因位置注入失败")
            print(f"  var 列: {list(adata.var.columns)}")
            print(f"  var_names 样例: {_sample_ids}")
    else:
        print(f"基因 ID 列: {_id_col}")

    if _id_col is not None:
        _gene_ids_raw = (
            adata.var_names.tolist() if _id_col == "_index"
            else adata.var[_id_col].astype(str).tolist()
        )
        import re
        _gene_ids_clean = [re.sub(r"\.\d+$", "", str(g)) for g in _gene_ids_raw]

        try:
            import mygene
            mg = mygene.MyGeneInfo()
            _batch_size = 1000
            _pos_records = []
            for _i in range(0, len(_gene_ids_clean), _batch_size):
                _batch = _gene_ids_clean[_i:_i + _batch_size]
                _results = mg.querymany(
                    _batch,
                    scopes="ensembl.gene",
                    fields="genomic_pos",
                    species="human",
                    as_dataframe=True,
                    returnall=False,
                )
                if isinstance(_results, pd.DataFrame) and len(_results) > 0:
                    _results.index.name = None
                    _pos_records.append(_results)

            if _pos_records:
                _pos_df = pd.concat(_pos_records)
                if "genomic_pos.chr" in _pos_df.columns:
                    _gene_pos = pd.DataFrame({
                        "query": _pos_df.index.astype(str).str.replace(r"\.\d+$", "", regex=True),
                        "chromosome": _pos_df["genomic_pos.chr"].astype(str),
                        "start": pd.to_numeric(_pos_df["genomic_pos.start"], errors="coerce"),
                        "end": pd.to_numeric(_pos_df["genomic_pos.end"], errors="coerce"),
                    })
                    _gene_pos = _gene_pos.reset_index(drop=True)
                    # 去重：mygene 可能对同一基因返回多处注释（如基因重复区域）
                    # 去重：同一基因多条 mygene 结果均映射至相同染色体，keep=first 安全
                    _gene_pos = _gene_pos.drop_duplicates(subset="query", keep="first")
                    print(f"  mygene 获取到 {len(_gene_pos)} 个基因的位置信息")
                    _success = True
                else:
                    print("  mygene 返回结果中无 genomic_pos 字段，查询失败")
                    _success = False
            else:
                print("  mygene 查询无结果")
                _success = False
        except ImportError:
            print("  mygene 未安装")
            _success = False
        except Exception as _e:
            print(f"  mygene 查询失败: {_e}")
            _success = False

        if _success and len(_gene_pos) > 0:
            _var_clean = adata.var.copy()
            _var_clean["_query_id"] = [
                re.sub(r"\.\d+$", "", str(g))
                for g in (_var_clean.index if _id_col == "_index"
                          else _var_clean[_id_col].astype(str))
            ]
            _merged = _var_clean.merge(
                _gene_pos,
                left_on="_query_id",
                right_on="query",
                how="left",
            )
            _merged.index = adata.var.index
            for _c in ["chromosome", "start", "end"]:
                adata.var[_c] = _merged[_c] if _c in _merged.columns else np.nan
            adata.var.drop(columns=["_query_id"], errors="ignore", inplace=True)
            print(f"位置列已注入 var: chromosome/start/end")
        else:
            print("位置注入失败——后续 CNV 推断将跳过")
else:
    print("位置列已存在，跳过注入")

In [ ]:
# 清洗染色体/坐标格式，过滤非标准染色体。
# 逻辑复用自 student-code 4.2/4.3（重写，去硬编码+补充解释）。
# 为什么过滤？CNV 是染色体臂级别的信号，性染色体和线粒体的覆盖模式不同；
# 只保留 1-22 + X/Y 染色体的基因，丢弃 mtDNA 和未映射的 contig。

if "chromosome" in adata.var.columns:
    # 统一染色体命名格式为 "chrN"
    adata.var["chromosome"] = (
        adata.var["chromosome"]
        .astype(str)
        .str.strip()
        .str.replace("^chr", "", regex=True)      # 去 chr 前缀（统一后再加回）
        .str.replace("^CHR", "", regex=True)
        .str.replace("^Chr", "", regex=True)
    )
    # 标记为 "chrN" 格式
    adata.var["chromosome"] = adata.var["chromosome"].apply(
        lambda x: f"chr{x}" if x in (["X", "Y"] + [str(i) for i in range(1, 23)]) else x
    )

    # 坐标转数值
    adata.var["start"] = pd.to_numeric(adata.var["start"], errors="coerce")
    adata.var["end"]   = pd.to_numeric(adata.var["end"], errors="coerce")

    # 只保留标准染色体 + 有效坐标
    _valid_chr = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
    _pos_mask = (
        adata.var["chromosome"].isin(_valid_chr)
        & adata.var["start"].notna()
        & adata.var["end"].notna()
        & (adata.var["start"] > 0)
        & (adata.var["end"] > adata.var["start"])
    )
    _n_before = adata.n_vars
    adata = adata[:, _pos_mask].copy()
    print(f"位置过滤: {_n_before} → {adata.n_vars} 基因 "
          f"（仅保留 1-22 + X/Y 染色体，坐标有效）")

    # 每条染色体至少保留 CNV_MIN_CHR_GENES 个基因
    _chr_counts = adata.var["chromosome"].value_counts()
    _keep_chr = _chr_counts[_chr_counts >= CNV_MIN_CHR_GENES].index
    _chr_mask = adata.var["chromosome"].isin(_keep_chr)
    if not _chr_mask.all():
        _n_before2 = adata.n_vars
        adata = adata[:, _chr_mask].copy()
        print(f"染色体过滤（<{CNV_MIN_CHR_GENES} 基因）: {_n_before2} → {adata.n_vars} 基因")
        _dropped = _chr_counts[_chr_counts < CNV_MIN_CHR_GENES]
        if len(_dropped) > 0:
            print(f"  丢弃染色体: {dict(_dropped)}")

    # 去重 —— 同一染色体坐标只保留第一条
    _dup_mask = adata.var.duplicated(subset=["chromosome", "start", "end"], keep="first")
    if _dup_mask.any():
        adata = adata[:, ~_dup_mask].copy()
        print(f"坐标去重: 移除 {_dup_mask.sum()} 个重复位置基因")

    # 染色体内按坐标排序 —— infercnv 要求基因按染色体位置有序
    adata = adata[:, adata.var.sort_values(["chromosome", "start", "end"]).index].copy()

    # 最终统计
    print(f"\nCNV 可用基因: {adata.n_vars}")
    print(adata.var["chromosome"].value_counts().sort_index().to_string())
else:
    print("chromosome 列不存在，CNV 无法继续——请先完成基因位置注入")

## 选择参考正常细胞

CNV 推断需要一组正常细胞作为基线——infercnv 计算每个基因组 bin 在所有
参考细胞中的平均表达，然后将每个细胞的信号与这个基线做差，检测偏离。

**为什么参考细胞的选择至关重要？**
如果错误地把肿瘤细胞纳入参考（例如上皮细胞中混杂了恶性细胞），
CNV 信号会被基线吸收——因为基线本身已经包含了异常拷贝数信号，
待检测细胞与基线做差后偏离不显著，**导致假阴性：
真正的恶性细胞被错误地判定为正常**。

**为什么选免疫/间质细胞？**
这些细胞类型通常不携带肿瘤特异性的大片段 CNV，是安全的参考基线。
上皮细胞中可能混杂肿瘤细胞，不适合做参考。
默认的参考细胞类型假设为非恶性，PI 应根据实际数据中的细胞类型分布
调整 `REFERENCE_CELL_TYPES`——例如已知数据中某上皮亚群确实是正常上皮，
可以将其加入参考列表以获得更贴近目标组织的基线。


In [ ]:
# 标记参考细胞——基于 cell_type_final_v1 选择非恶性细胞类型。
# 为什么选免疫/间质细胞？它们通常不携带肿瘤特异性的大片段 CNV；
# 上皮细胞中可能混杂肿瘤细胞，不适合做参考基线。
if REFERENCE_KEY in adata.obs.columns:
    _avail_ct = set(adata.obs[REFERENCE_KEY].dropna().astype(str).unique())
    _ref_ct = [ct for ct in REFERENCE_CELL_TYPES if ct in _avail_ct]
    _non_ref_ct = list(_avail_ct - set(_ref_ct))

    print(f"参考细胞类型 ({len(_ref_ct)}): {_ref_ct}")
    print(f"非参考细胞类型 ({len(_non_ref_ct)}): {_non_ref_ct}")

    adata.obs["cnv_reference"] = adata.obs[REFERENCE_KEY].astype(str).isin(_ref_ct).astype(str)
    adata.obs["cnv_reference"] = adata.obs["cnv_reference"].astype("category")

    _n_ref = (adata.obs["cnv_reference"] == "True").sum()
    _n_non = (adata.obs["cnv_reference"] == "False").sum()
    print(f"参考细胞: {_n_ref}  |  待检测细胞: {_n_non}")

    if _n_ref == 0:
        print("WARNING: 没有匹配的参考细胞类型！请调整 REFERENCE_CELL_TYPES。")
    elif _n_non == 0:
        print("WARNING: 没有非参考细胞！所有细胞都匹配了参考类型——请调整。")
else:
    print(f"REFERENCE_KEY '{REFERENCE_KEY}' 不在 obs 列中，无法标记参考细胞")
    adata.obs["cnv_reference"] = "unknown"  # 预建空列

## CNV 推断：从表达量到拷贝数变异的信号转换

调用 `cnv.tl.infercnv`，核心算法分三步：

**第一步——滑动窗口平滑**
沿每条染色体取 `window_size=100` 个基因做滑动平均（`step=10`）。
单个基因的表达有大量技术噪声和生物学随机波动，
但 CNV 是染色体臂级别的大规模事件，会影响整段数百个基因。
滑动平均把单个基因的噪声压下去，把染色体区域级别的系统性偏离信号提上来。

**第二步——参考基线对比**
每个窗口用参考正常细胞的平均表达做基线，
计算待检测细胞在该窗口的表达偏离（log fold change）。
这就是为什么参考细胞的选择至关重要：基线错了，所有偏离都不可信。

**第三步——logFC 裁剪**
`lfc_clip=3.0` 限制极端偏离值，避免个别极端高表达基因
（如免疫球蛋白基因在浆细胞中的极端表达）支配整个 CNV 信号。

完成后对 `X_cnv` 矩阵做 PCA -> 邻居图 -> Leiden 聚类，
产生一个**基于 CNV 信号（而非转录组相似性）的细胞聚类**——
这个聚类本身就是恶性判断的重要线索：
如果某个 Leiden 簇富集了高 CNV 信号的细胞，
该簇很可能是恶性/癌前病变群体。


In [ ]:
# CNV 推断——infercnvpy 核心步骤。
if "cnv_reference" in adata.obs.columns and adata.obs["cnv_reference"].nunique() > 1:
    _ref_cat = ["True"]  # 参考细胞标记值在 cnv_reference 列中为 "True"

    print(f"CNV 推断参数: window_size={CNV_WINDOW_SIZE}, step={CNV_STEP}, "
          f"lfc_clip={CNV_LFC_CLIP}")
    print(f"参考细胞标记: cnv_reference='True' ({_ref_cat})")

    # 用 copy 做 CNV（避免污染原 adata）
    adata_cnv = adata.copy()

    cnv.tl.infercnv(
        adata_cnv,
        reference_key="cnv_reference",
        reference_cat=_ref_cat,
        window_size=CNV_WINDOW_SIZE,
        step=CNV_STEP,
        dynamic_threshold=None,   # 不自动设参考阈值——已手动指定
        lfc_clip=CNV_LFC_CLIP,
        n_jobs=1,                 # 单线程避免 Jupyter 环境死锁
    )
    print(f"infercnv 完成: X_cnv shape={adata_cnv.obsm['X_cnv'].shape}")

    # PCA → 邻居图 → Leiden（基于 CNV 信号的降维与聚类）
    cnv.tl.pca(adata_cnv)
    cnv.pp.neighbors(adata_cnv)
    cnv.tl.leiden(adata_cnv, resolution=0.5)
    print(f"CNV Leiden 聚类: {adata_cnv.obs['cnv_leiden'].nunique()} 簇")

    # 把 CNV 相关结果写回原 adata
    adata.obsm["X_cnv"] = adata_cnv.obsm["X_cnv"]
    adata.obsm["X_cnv_pca"] = adata_cnv.obsm["X_cnv_pca"]
    adata.obs["cnv_leiden"] = adata_cnv.obs["cnv_leiden"].values
    print("CNV 推断结果已写回 adata")
else:
    adata_cnv = None
    print("参考细胞未标记或仅有一种类型，CNV 推断跳过")

## CNV 评分：量化每个细胞的基因组不稳定性

`cnv.tl.cnv_score` 对每个细胞计算 CNV 信号强度的标量评分——
即该细胞在所有基因组窗口上偏离参考基线的绝对程度的总和。
**评分越高 = 该细胞的大片段 CNV 越多/越显著 = 越可能是恶性/异常细胞**。

分别在两个维度上汇总评分：
- **CNV Leiden 簇**：看基于 CNV 信号自己聚出来的簇中，哪些簇整体 CNV 偏高
- **原始细胞类型**：看原始注释的各类细胞中，哪类细胞的 CNV 信号异常

**临床判读**：上皮细胞整体 cnv_score 偏高 -> 提示恶性上皮群体存在；
成纤维细胞 cnv_score 整体低 -> 符合预期（间质细胞通常无不稳定性）；
如果免疫细胞中也出现高 cnv_score 的子群 -> 需警惕技术假象或罕见事件。


In [ ]:
# CNV 评分——分两个维度计算。
if adata_cnv is not None:
    # 维度 1: 按 CNV Leiden 簇（基于 CNV 信号自己聚的簇）
    cnv.tl.cnv_score(adata_cnv, groupby="cnv_leiden")
    adata.obs["cnv_score"] = adata_cnv.obs["cnv_score"].values

    # 维度 2: 按原始细胞类型（看哪类细胞 CNV 信号强）
    if REFERENCE_KEY in adata.obs.columns:
        cnv.tl.cnv_score(adata_cnv, groupby=REFERENCE_KEY)
        adata.obs[f"cnv_score_by_{REFERENCE_KEY}"] = adata_cnv.obs["cnv_score"].values

    print("cnv_score 已计算")
    print(f"  mean cnv_score: {adata.obs['cnv_score'].mean():.4f}")
    print(f"  cnv_score 范围: [{adata.obs['cnv_score'].min():.4f}, "
          f"{adata.obs['cnv_score'].max():.4f}]")

    # CNV 结果已全部写回 adata（X_cnv、X_cnv_pca、cnv_leiden、cnv_score），
    # 立即释放 adata_cnv——infercnv 的 dense obsm["X_cnv"] 非常占内存，
    # 拖到 notebook 末尾会不必要地挤占后续可视化阶段的 RAM。
    del adata_cnv
    gc.collect()
    adata_cnv = None  # 置 None 保持后续 if adata_cnv is not None 守卫语义一致
    print("adata_cnv 已释放（CNV 结果已全量写回 adata）")
else:
    print("adata_cnv 为 None，跳过 CNV 评分")

## 染色体级 CNV heatmap

`X_cnv` 矩阵（细胞数 x 基因组窗口数）展示了每个细胞在每条染色体上的
拷贝数偏离程度。**红色 = 扩增区域（表达高于参考基线），
蓝色 = 缺失区域（表达低于参考基线）**。

**为什么采样子集？**
全量矩阵（数千细胞 x 上万窗口）渲染成本高、肉眼无法分辨单个细胞，
此处对代表性细胞子集（<=500 个）采样作图，按 `cnv_score` 降序排列——
高 CNV 信号的细胞排在上方，让异常群体一目了然。

**怎么看？**
- 上方细胞整行偏红 -> 广泛扩增，高度提示恶性
- 上方细胞某些染色体区域红、其余中性 -> 局灶性 CNV，可能有亚克隆
- 下方细胞（低 cnv_score）整体接近白色 -> 正常二倍体，参考基线附近
- 某条染色体在所有细胞中都有异常信号 -> 可能是生殖系 CNV 或技术偏差

按 `cnv_leiden` 分组着色（右侧色条），看哪些 CNV 簇的染色体谱系相似。


In [ ]:
# 可视化 1: 染色体级 CNV heatmap。
# X_cnv 矩阵太大（细胞数 × 窗口数），只对代表性细胞子集画 heatmap。
# 为什么采样子集？全量热图（几千细胞 × 上万窗口）无法在 notebook 中渲染。
# 注意：adata_cnv 已在 cnv_score cell 后释放——此处用 adata（CNV 结果已全量写回）。
if "X_cnv" in adata.obsm:
    _n_sample = min(500, adata.n_obs)
    _rng = np.random.default_rng(42)
    _sample_idx = _rng.choice(adata.n_obs, size=_n_sample, replace=False)
    _cnv_sub = adata[_sample_idx].copy()

    # 按 cnv_score 排序细胞（高 CNV 信号在上方）
    _cnv_sub = _cnv_sub[_cnv_sub.obs["cnv_score"].argsort()[::-1]].copy()

    try:
        fig = cnv.pl.chromosome_heatmap(
            _cnv_sub,
            groupby="cnv_leiden",
            show_gene_labels=False,
            figsize=(16, max(4, _n_sample * 0.05)),
            show=False,
        )
        _hm_path = "results/figures/09_cnv_heatmap.png"
        fig.savefig(_hm_path, dpi=200, bbox_inches="tight")
        plt.close("all")
        print(f"CNV heatmap 已保存: {_hm_path} ({_n_sample} 个采样细胞)")
    except Exception as _e:
        print(f"CNV heatmap 绘制失败: {_e}")
        print("这通常是因为 X_cnv 的染色体标注与 infercnvpy 期望不一致，")
        print("不影响 adata.uns 中的 CNV 结果数据。")

    del _cnv_sub
    gc.collect()
else:
    print("X_cnv 不在 adata.obsm 中，跳过 heatmap")

## UMAP 着色：CNV 信号在细胞图谱上的分布

在原始 UMAP（来自 04 embedding）上用 `cnv_score` 和 `cnv_leiden` 着色，
**直接回答：在细胞图谱上哪些区域的 CNV 信号偏高？**

- `cnv_score`：coolwarm 色阶，蓝 = 正常（低 CNV），红 = 异常（高 CNV）
- `cnv_leiden`：基于 CNV 信号的 Leiden 聚类标签

**临床判读**：
- 上皮细胞区域出现集中的高 cnv_score -> 高度提示恶性/癌前病变群体
- 免疫/间质区域整体低 cnv_score -> 符合预期（非恶性细胞通常无大片段 CNV）
- cnv_leiden 的高分簇与原始细胞类型的空间关系 ->
  看恶性群体是集中在上皮区域（典型癌）还是散布多处（需警惕）。


In [ ]:
# 可视化 2: UMAP 着色——cnv_score + cnv_leiden。
# 用原始 04/06 的 UMAP 坐标（不是 CNV 空间的），
# 这样可以直接看"在原始细胞图谱上哪些区域 CNV 信号高"。
if "X_umap" in adata.obsm:
    if "cnv_score" in adata.obs.columns:
        # cnv_score: 颜色映射（低=正常=蓝，高=异常=红）
        sc.pl.umap(
            adata,
            color="cnv_score",
            cmap="coolwarm",
            title="CNV score (per cell)",
            show=False,
            save="_09_cnv_score.png",
        )
        _src = "figures/umap_09_cnv_score.png"
        _dst = "results/figures/09_cnv_umap_score.png"
        os.makedirs(os.path.dirname(_dst), exist_ok=True)
        if os.path.exists(_src):
            os.rename(_src, _dst)
            print(f"CNV score UMAP 已保存: {_dst}")
        else:
            print(f"WARNING: UMAP 图未产出: {_src}")

        sc.pl.umap(
            adata,
            color="cnv_leiden",
            title="CNV-based Leiden clusters",
            legend_loc="right margin",
            show=False,
            save="_09_cnv_leiden.png",
        )
        _src2 = "figures/umap_09_cnv_leiden.png"
        _dst2 = "results/figures/09_cnv_umap_leiden.png"
        os.makedirs(os.path.dirname(_dst2), exist_ok=True)
        if os.path.exists(_src2):
            os.rename(_src2, _dst2)
            print(f"CNV Leiden UMAP 已保存: {_dst2}")
        else:
            print(f"WARNING: UMAP 图未产出: {_src2}")

        plt.close("all")
    else:
        print("cnv_score 不在 adata.obs 中，跳过 UMAP")
else:
    print("X_umap 不在 obsm 中——可能上游没有跑 UMAP。请手动 sc.tl.umap() 或回 04 补充。")


## 结果保存与追溯

本 stage 写入 `adata` 的核心结果字段：
- CNV 信号矩阵 -> `adata.obsm["X_cnv"]`（细胞 x 基因组窗口）
- CNV PCA -> `adata.obsm["X_cnv_pca"]`
- CNV 评分 -> `adata.obs["cnv_score"]`（每细胞一个标量）
- CNV Leiden 聚类 -> `adata.obs["cnv_leiden"]`
- 运行元数据 -> `adata.uns["09_cnv_v1"]`（参数 + 时间戳）
- 追踪字段 -> `adata.uns["stage"]` / `version` / `upstream` / `status`
- 产出 h5ad -> `OUTPUT_PATH`


In [ ]:
# 写入 adata.uns 元数据 + 导出 CSV。
import datetime as _dt

_cnv_score_summary = None
if "cnv_score" in adata.obs.columns and REFERENCE_KEY in adata.obs.columns:
    _cnv_score_summary = (
        adata.obs.groupby(REFERENCE_KEY)["cnv_score"]
        .agg(["mean", "std", "count"])
        .round(4)
    )
    _cnv_score_summary.columns = ["cnv_score_mean", "cnv_score_std", "n_cells"]
    _cnv_csv = "results/tables/09_cnv_score_by_celltype.csv"
    _cnv_score_summary.to_csv(_cnv_csv)
    print(f"cnv_score 按细胞类型汇总 → {_cnv_csv}")
    print(_cnv_score_summary.to_string())

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 命名一致）
adata.uns["stage"] = "09_cnv"     # 本 stage 标识
adata.uns["version"] = "v1"                  # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"


adata.uns["09_cnv_v1"] = {
    "method": "infercnvpy (pure Python)",
    "window_size": CNV_WINDOW_SIZE,
    "step": CNV_STEP,
    "lfc_clip": CNV_LFC_CLIP,
    "min_chr_genes": CNV_MIN_CHR_GENES,
    "reference_key": REFERENCE_KEY,
    "reference_cell_types": REFERENCE_CELL_TYPES,
    "cnv_score_csv": "results/tables/09_cnv_score_by_celltype.csv" if _cnv_score_summary is not None else None,
    "timestamp": _dt.datetime.now().isoformat(),
}
print("运行元数据已写入 adata.uns['09_cnv_v1']")

In [ ]:
# 内存自检。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 检查 X_cnv 是否有异常大小
if "X_cnv" in adata.obsm:
    _cnv_shape = adata.obsm["X_cnv"].shape
    _cnv_mb = adata.obsm["X_cnv"].nbytes / 1e6
    print(f"X_cnv shape: {_cnv_shape}  ({_cnv_mb:.1f} MB)")

In [ ]:
# 写出 checkpoint。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 释放内存。
# 注意：adata_cnv 已在 cnv_score cell 后提前释放，这里不再重复 del。
del adata
gc.collect()
print("内存已释放")
